# Tutorial notebook 7: Custom prompt templates

In this tutorial, we will show how to use custom prompt templates for finetuning Cell2Sentence models on your own tasks and datasets. This may be needed if you want to use Cell2Sentence for a task that is not one of the standard tasks supported by Cell2Sentence.

At a high level, we will:
1. Write a custom prompt template
2. Subclass the PromptFormatter class to implement a function that formats each cell sentence sample in our dataset with our custom prompt template.
3. Finetune a Cell2Sentence model on our custom formatted dataset.

We will use the same dataset and model as in the previous tutorials.

Import necessary libraries

In [ ]:
# Python built-in libraries
import os
import pickle
import random
from datetime import datetime
from collections import Counter

# Third-party libraries
from datasets import Dataset
import numpy as np
from tqdm import tqdm
from transformers import TrainingArguments

# Single-cell libraries
import anndata
import scanpy as sc

# Cell2Sentence imports
import cell2sentence as cs
from cell2sentence.prompt_formatter import get_cell_sentence_str
from cell2sentence.tasks import predict_cell_types_of_data


In [ ]:
from cell2sentence.prompt_formatter import PromptFormatter

In [ ]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)

# Load Data

Next, we will load the preprocessed dataset from the tutorial 0. This dataset has already been filtered and normalized, so it it ready for transformation into cell sentences.

<font color='red'>Please make sure you have completed the preprocessing steps in Tutorial 0 before running the following code, if you are using your own dataset.</font>. Ensure that the file path is correctly set in <font color='gold'>DATA_PATH</font> to where your preprocessed data was saved from tutorial 0.

In [ ]:
DATA_PATH = "/home/sr2464/scratch/C2S_Files/Cell2Sentence_Datasets/dominguez_conde_immune_tissue_two_donors_preprocessed_tutorial_0.h5ad"

In [ ]:
adata = anndata.read_h5ad(DATA_PATH)
adata

In [ ]:
adata.obs = adata.obs[["cell_type", "tissue", "batch_condition", "organism", "sex"]]

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type",
    size=8,
    title="Human Immune Tissue UMAP",
)

In [ ]:
adata.X.max()

We are expecting log10 base 10 transformed data, with a maximum value somewhere around 3 or 4. Make sure to start with processed and normalized data when doing the cell sentence conversion!

# Cell2Sentence Conversion

In this section, we will transform our AnnData object containing our single-cell dataset into a Cell2Sentence (C2S) dataset by calling the functions of the CSData class in the C2S code base. Full documentation for the functions of the CSData class can be found in the documentation page of C2S.

In [ ]:
adata_obs_cols_to_keep = adata.obs.columns.tolist()
adata_obs_cols_to_keep

In [ ]:
# Create CSData object
arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
    adata=adata, 
    random_state=SEED, 
    sentence_delimiter=' ',
    label_col_names=adata_obs_cols_to_keep
)

In [ ]:
arrow_ds

In [ ]:
sample_idx = 0
arrow_ds[sample_idx]

# Custom Prompt Formatting

Here, we will define a custom prompt template for a new task which we will finetune a Cell2Sentence model on. For the purposes of this tutorial, we will define a task where the Cell2Sentence model is given a cell sentence and the tissue which a cell originates from, and the model must predict the cell type of the cell. This is a variation of the standard cell type prediction task supported by Cell2Sentence, but it demonstrates that we can modify the input prompt of Cell2Sentence to include new information and rearrange the input format for new tasks.

The keywords enclosed in curly braces `{}` will be replaced with the actual values when the prompt template is applied to a cell sentence sample.

In [ ]:
custom_input_prompt_template = """Given below is a list of {num_genes} gene names ordered by descending expression level in a {organism} cell. Given this expression representation as well as the tissue which this cell originates from, your task is to give the cell type which this cell belongs to.\nTissue type: {tissue_type}\nCell sentence: {cell_sentence}.\nThe cell type corresponding to these genes is:"""
answer_template = "{cell_type}"

Here, we create a subclass of the PromptFormatter class. The PromptFormatter class is responsible for formatting each cell sentence sample in our dataset with a prompt template. It defines a function called `format_hf_ds` that takes in a cell sentence arrow dataset and returns a formatted dataset, applying the prompt template to each sample. We will implement our custom formatting logic in this `format_hf_ds` function.

You can see the original PromptFormatter class [here](https://github.com/vandijklab/cell2sentence/blob/custom_formatter/src/cell2sentence/prompt_formatter.py).

In [ ]:
class CustomPromptFormatter(PromptFormatter):
    def __init__(self, task_name, input_prompt, answer_template, top_k_genes):
        super().__init__()
        self.task_name = task_name
        self.input_prompt = input_prompt
        self.answer_template = answer_template
        self.top_k_genes = top_k_genes
        assert top_k_genes > 0, "'top_k_genes' must be an integer > 0"

    def format_hf_ds(self, hf_ds):
        """
        Our custom formatting function which will loop through the dataset samples and format
        cell sentences and cell metadata into the custom prompt template.

        Arguments:
            hf_ds: Huggingface arrow dataset containing cell sentences to format prompts with.
        
        Returns:
            ds: Huggingface dataset containing formatted cell sentences and cell metadata.
                Dataset columns expected: 'sample_type', 'model_input', 'response'
        """
        # Initialize lists to store formatted model inputs and responses
        model_inputs_list = []
        responses_list = []

        # Loop through each sample in the dataset
        for cell_idx in range(hf_ds.num_rows):
            # Get the sample from the arrow dataset - contains cell sentence and cell metadata
            sample = hf_ds[cell_idx]
            
            # Get cell sentence
            single_cell_sentence_str, num_genes_str = get_cell_sentence_str(sample, num_genes=self.top_k_genes)
            model_input_str = self.input_prompt.format(
                cell_sentence=single_cell_sentence_str,
                num_genes=num_genes_str,
                organism=sample["organism"],
                tissue_type=sample["tissue"],
            )
            response_str = self.answer_template.format(
                cell_type=sample["cell_type"],
            )

            model_inputs_list.append(model_input_str)
            responses_list.append(response_str)

        # Create formatted Huggingface dataset
        ds_split_dict = {
            "sample_type": [self.task_name] * hf_ds.num_rows,
            "model_input": model_inputs_list,
            "response": responses_list,
        }
        ds = Dataset.from_dict(ds_split_dict)
        return ds

Now we define an instance of our CustomPromptFormatter class.

In [ ]:
task_name = "cell_type_pred_given_tissue"
prompt_formatter = CustomPromptFormatter(
    task_name=task_name,
    input_prompt=custom_input_prompt_template,
    answer_template=answer_template,
    top_k_genes=100
)
prompt_formatter

Let's format a few dataset samples with our custom prompt template to see what the formatted dataset looks like.

In [ ]:
small_example_ds = arrow_ds.select(range(10))
small_example_ds

In [ ]:
formatted_small_example_ds = prompt_formatter.format_hf_ds(small_example_ds)
formatted_small_example_ds

In [ ]:
formatted_small_example_ds[0]

In [ ]:
print("#----Model input:----#")
print(formatted_small_example_ds[0]["model_input"], "\n")
print("#----Response:----#")
print(formatted_small_example_ds[0]["response"])

We can see that the model input is formatted with the custom prompt template, and the response is the cell type annotation for the cell sentence sample. One thing to note here is that in the current format, all samples will have the exact same input template, so there will not be variations in natural language of how we ask the LLM to perform the task. It can be beneficial to provide variations of prompt templates to provide some diversity - simply create several templates and choose one when formatting each sample in the formatting function!

For our finetuning run, the csmodel finetune() function will do the formatting on the full dataset for us, so we do not need to do it manually. We will simply pass in our arrow dataset as well as our CustomPromptFormatter instance to the finetune() function.

# Create CSData object

As one last data step, we need to create a CSData object which will wrap around our arrow dataset.

In [ ]:
c2s_save_dir = "/home/sr2464/scratch/C2S_Files/c2s_api_testing"  # C2S dataset will be saved into this directory
c2s_save_name = "dominguez_immune_tissue_tutorial7"  # This will be the name of our C2S dataset on disk

In [ ]:
csdata = cs.CSData.csdata_from_arrow(
    arrow_dataset=arrow_ds, 
    vocabulary=vocabulary,
    save_dir=c2s_save_dir,
    save_name=c2s_save_name,
    dataset_backend="arrow"
)

In [ ]:
print(csdata)

# Load Cell2Sentence Model

Now, we will load a C2S model which will finetune on a new dataset, as in tutorial 3. This model can be a LLM pretrained on natural language, or it can be a trained C2S model which will undergo further finetuning on a new dataset of interest. Typically, starting from a pretrained C2S model benefits performance, since C2S models were initialized from natural language-pretrained LLMs and trained on many single-cell datasets on different tasks.

For this tutorial, we will start finetuning from the C2S-Pythia-410M cell type prediction model, which was trained to do cell type prediction on many datasets from CellxGene and Human Cell Atlas. More details about the C2S-Pythia-410M cell type prediction model can be found in the Model Zoo section of the ReadME in the GitHub repo, or in the Huggingface model card.

We can define our CSModel object with our pretrained cell type prediction model as follows:

In [ ]:
# Define CSModel object
model_name_or_path = "vandijklab/C2S-Pythia-410m-diverse-single-and-multi-cell-tasks"
save_dir = "/home/sr2464/scratch/C2S_Files/c2s_api_testing/csmodel_tutorial_7"
save_name = "cell_type_pred_pythia_410M_2"
csmodel = cs.CSModel(
    model_name_or_path=model_name_or_path,
    save_dir=save_dir,
    save_name=save_name
)

In [ ]:
print(csmodel)

# Finetune on new dataset

Now, we will finetune our C2S model on our custom task. The finetune() function will format the full dataset for us using our CustomPromptFormatter instance, so we just need to pass in our custom prompt formatter to the finetune() function.

For training, we will need to define training arguments for finetuning our C2S model on our new dataset. Huggingface's Trainer class is used to do training, so we can utilize different training techniques (e.g. mixed precision training, gradient accumulation, gradient checkpointing, etc.) by specifying the corresponding option in the TrainingArguments object. This gives us a vast array of possible options for training, and will allow us to specify important parameters such as batch size, learning rate, and learning rate schedulers. See the full documentation for training arguments at:
- https://huggingface.co/docs/transformers/en/main_classes/trainer

In [ ]:
task_name

In [ ]:
datetimestamp = datetime.now().strftime('%Y-%m-%d-%H_%M_%S')
output_dir = os.path.join(csmodel.save_dir, datetimestamp + f"_finetune_{task_name}")
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
print(output_dir)

Here, we define our training arguments. For this tutorial, we will use a batch size of 8 with 4 gradient accumulation steps, yielding an effective batch size of 32. We will use a learning rate of 1e-5 with a cosine annealing scheduler, and we will train for 5 epochs total. Some other important parameters specified here are:
- bf16: Uses mixed-precision training with bfloat16 dtype
- logging_steps: controls how often we log training loss
- eval_steps: controls how often we run the eval loop
- warmup_ratio: percentage of training in which learning rate warms up to the base learning rate specified

Full explanations of all possible training arguments can be found in the Huggingface Trainer documentation: 

https://huggingface.co/docs/transformers/v4.44.2/en/main_classes/trainer#transformers.TrainingArguments

In [ ]:
train_args = TrainingArguments(
    bf16=True,
    fp16=False,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    learning_rate=1e-5,
    load_best_model_at_end=True,
    logging_steps=50,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    num_train_epochs=5, 
    eval_steps=50,
    evaluation_strategy="steps",
    save_steps=100,
    save_strategy="steps",
    save_total_limit=3,
    warmup_ratio=0.05,
    output_dir=output_dir,
    max_steps=1000  # Set a maximum number of steps - shortened to 1k steps for tutorial purposes
)

In [ ]:
csmodel.fine_tune(
    csdata=csdata,
    task=task_name,
    train_args=train_args,
    loss_on_response_only=False,
    top_k_genes=100,
    max_eval_samples=500,
    prompt_formatter=prompt_formatter  # Pass in our custom prompt formatter
)